# 🧭 Career Compass — AI Job Finder Agent

A tiny LangGraph agent that routes a person from **degree → career category → real job/skill/company recommendations**, with a Groq LLM as a fallback router for degrees that don't match an obvious keyword.

**Flow:** `intake → router → (engineering_tech | business_management | medical_health | arts_humanities) → job_lookup → END`

This notebook is the same graph as the original — the upgrades are: a course **selection menu** (a dropdown-style picker since plain input() can't render an HTML `<select>`), fuzzy matching so near-miss spellings still resolve, and a more characterful printed report at the end. This same agent also powers a full web app — see the `frontend/` and `backend/` folders shipped alongside this notebook if you want a real UI.

## 1. Install & import

In [ ]:
!pip install -qU langgraph langchain-groq pandas
import os
import io
import re
import difflib
import pandas as pd
from getpass import getpass
from typing import TypedDict, Literal
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END


## 2. Groq API key

Get a free key at https://console.groq.com/keys

In [ ]:
os.environ["GROQ_API_KEY"] = getpass("🔑 Enter your Groq API key: ")  # never shared, never printed


## 3. Load the degree → jobs dataset

Upload `degree_to_jobs_dataset.csv` (shipped alongside this notebook) when prompted. It now also carries a `Vibe` tag and a 3-step `Roadmap` for each degree, which the final report uses.

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick degree_to_jobs_dataset.csv
degree_df = pd.read_csv("degree_to_jobs_dataset.csv")
print(f"📚 Loaded {len(degree_df)} degrees across {degree_df['Category'].nunique()} career categories.")
degree_df.head()


## 4. Shared memory (agent state)

In [ ]:
class JobSeekerState(TypedDict, total=False):
    name: str
    degree: str
    interests: str
    category: str      # Routed category (e.g., engineering_tech)
    reasoning: str
    vibe: str
    recommendations: dict


## 5. Intake node — course "dropdown"

Colab can't render a real HTML `<select>` in a plain `input()` prompt, so this is the closest equivalent: a numbered menu built straight from the dataset, grouped by category — pick a number, or type `0` to enter a degree that isn't listed. (The hosted web app in `frontend/` renders this as an actual `<select>` dropdown.)

In [ ]:
CATEGORY_LABELS = {
    "engineering_tech": "🖥️  Engineering & Tech",
    "business_management": "💼  Business & Management",
    "medical_health": "🩺  Medical & Health",
    "arts_humanities": "🎨  Arts & Humanities",
}

def print_course_menu(df: pd.DataFrame) -> list:
    """Prints a numbered, category-grouped menu and returns the ordered degree list
    so a typed number can be mapped back to a row — the notebook's stand-in for a dropdown."""
    ordered = []
    print("\n=== 🧭 Pick your course (type its number) ===")
    for cat, label in CATEGORY_LABELS.items():
        rows = df[df["Category"] == cat]
        if rows.empty:
            continue
        print(f"\n{label}")
        for _, row in rows.iterrows():
            ordered.append(row["Degree"])
            print(f"  [{len(ordered):>2}] {row['Degree']}")
    print(f"\n  [ 0] Something else (type it in manually)")
    return ordered

def intake_node(state: JobSeekerState) -> JobSeekerState:
    print("\n=== Career Advisor Intake ===")
    name = input("👋 Enter your name: ").strip() or "Explorer"

    course_order = print_course_menu(degree_df)
    choice = input("\nYour pick (number): ").strip()
    if choice.isdigit() and 1 <= int(choice) <= len(course_order):
        degree = course_order[int(choice) - 1]
        print(f"✅ Selected: {degree}")
    else:
        degree = input("What is your degree? (e.g., Computer Science, MBA): ").strip()

    interests = input("🎯 What are your key skills or interests? ").strip()
    return {
        "name": name,
        "degree": degree,
        "interests": interests,
        "category": "",
        "reasoning": "",
        "vibe": "",
        "recommendations": {},
    }


## 6. Router node

Keyword guardrails handle the obvious cases instantly; anything ambiguous goes to the Groq LLM for a judgment call.

In [ ]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

ROUTER_SYSTEM_PROMPT = """You are a career path router. Based on the user's degree,
classify them into EXACTLY ONE of these categories:
- engineering_tech
- business_management
- medical_health
- arts_humanities

Respond with ONLY one word from the list above."""

def router_node(state: JobSeekerState) -> JobSeekerState:
    degree_lower = state["degree"].lower()

    if any(kw in degree_lower for kw in ["software", "computer", "engineer", "it", "tech", "cyber", "data"]):
        category = "engineering_tech"
    elif any(kw in degree_lower for kw in ["business", "mba", "management", "finance", "marketing", "econom", "account"]):
        category = "business_management"
    elif any(kw in degree_lower for kw in ["medic", "mbbs", "nurs", "health", "pharma", "physio", "biomed"]):
        category = "medical_health"
    elif any(kw in degree_lower for kw in ["art", "design", "journal", "literat", "psycholog", "politic", "history", "sociol"]):
        category = "arts_humanities"
    else:
        # Ask the LLM for complex / unlisted degrees
        response = llm.invoke([
            SystemMessage(content=ROUTER_SYSTEM_PROMPT),
            HumanMessage(content=f"Degree: {state['degree']}\nInterests: {state['interests']}")
        ])
        category = response.content.strip().lower()

    # Normalize whatever came back to a graph path
    if "engineering" in category or "tech" in category: category = "engineering_tech"
    elif "business" in category or "management" in category: category = "business_management"
    elif "medical" in category or "health" in category: category = "medical_health"
    else: category = "arts_humanities"

    return {**state, "category": category, "reasoning": f"Degree '{state['degree']}' matched to {category}"}


## 7. Department nodes & job lookup

Each department node just sets the mood; `job_lookup_node` does the real work — an exact match against the dataset first, then a fuzzy fallback (so "Buisness Admin" still finds "Business Administration"), then a friendly generic fallback if nothing's close. It also now returns a 3-step roadmap and a vibe tag.

In [ ]:
def engineering_node(state: JobSeekerState) -> JobSeekerState:
    print(f"🛠️  Processing {state['name']} for Engineering & Tech roles...")
    return state

def business_node(state: JobSeekerState) -> JobSeekerState:
    print(f"💼 Processing {state['name']} for Management & Corporate roles...")
    return state

def medical_node(state: JobSeekerState) -> JobSeekerState:
    print(f"🏥 Processing {state['name']} for Healthcare roles...")
    return state

def arts_node(state: JobSeekerState) -> JobSeekerState:
    print(f"🎨 Processing {state['name']} for Creative & Social Science roles...")
    return state

def _best_row_for_degree(degree: str):
    match = degree_df[degree_df["Degree"].str.contains(re.escape(degree), case=False, na=False)]
    if not match.empty:
        return match.iloc[0], "exact"
    close = difflib.get_close_matches(degree, degree_df["Degree"].tolist(), n=1, cutoff=0.5)
    if close:
        return degree_df[degree_df["Degree"] == close[0]].iloc[0], "fuzzy"
    return None, "none"

def job_lookup_node(state: JobSeekerState) -> JobSeekerState:
    row, quality = _best_row_for_degree(state["degree"])

    if row is not None:
        recs = {
            "jobs": row["Suitable_Jobs"],
            "skills": row["Key_Skills"],
            "companies": row["Companies_To_Apply"],
            "roadmap": [s.strip() for s in row["Roadmap"].split("|")],
            "match_quality": quality,
        }
        vibe = row["Vibe"]
    else:
        recs = {
            "jobs": "No specific match found",
            "skills": "Communication; Adaptability; Digital Literacy",
            "companies": "Various startups",
            "roadmap": ["Talk to people in a field you admire", "Build one small project", "Try a short online course"],
            "match_quality": "none",
        }
        vibe = "🧭 The Explorer"

    return {**state, "recommendations": recs, "vibe": vibe}


## 8. Build & compile the graph

In [ ]:
def route_decision(state: JobSeekerState) -> Literal["engineering_tech", "business_management", "medical_health", "arts_humanities"]:
    return state["category"]

builder = StateGraph(JobSeekerState)

builder.add_node("intake", intake_node)
builder.add_node("router", router_node)
builder.add_node("engineering_tech", engineering_node)
builder.add_node("business_management", business_node)
builder.add_node("medical_health", medical_node)
builder.add_node("arts_humanities", arts_node)
builder.add_node("job_lookup", job_lookup_node)

builder.set_entry_point("intake")
builder.add_edge("intake", "router")

builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "engineering_tech": "engineering_tech",
        "business_management": "business_management",
        "medical_health": "medical_health",
        "arts_humanities": "arts_humanities",
    },
)

for node in ["engineering_tech", "business_management", "medical_health", "arts_humanities"]:
    builder.add_edge(node, "job_lookup")

builder.add_edge("job_lookup", END)
graph = builder.compile()


## 9. Visualize the graph

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


## 10. Run the agent 🚀

In [ ]:
final_state = graph.invoke({})

recs = final_state["recommendations"]
width = 58
print("\n" + "═" * width)
print(f"🎓 CAREER REPORT — {final_state['name'].upper()}")
print(f"{final_state['vibe']}")
print("─" * width)
print(f"Degree     : {final_state['degree']}")
print(f"Category   : {final_state['category'].replace('_', ' ').title()}")
print(f"Match type : {recs.get('match_quality', 'n/a')}")
print("─" * width)
print(f"🚀 Recommended Jobs   : {recs['jobs']}")
print(f"🛠️  Key Skills        : {recs['skills']}")
print(f"🏢 Companies to Target: {recs['companies']}")
print("─" * width)
print("🗺️  Your 3-step roadmap:")
for i, step in enumerate(recs.get("roadmap", []), start=1):
    print(f"   {i}. {step}")
print("═" * width)


---
### Want a real web app instead of typing into a Colab cell?

This exact agent (same state, same nodes, same edges) also runs behind a FastAPI backend with a proper HTML dropdown frontend — see the `backend/` and `frontend/` folders shipped next to this notebook, deployable to **Render** (API) and **Vercel** (UI), or runnable entirely on `localhost`.